# The Long Tail of Literature: Visual Story

This notebook creates **5 interactive visualizations** that tell the story of book popularity in the Open Library dataset.

## Design Principles
- **Maximize Data-Ink Ratio** (Tufte): Clean, minimal design with `simple_white` template
- **Reduce Clutter** (Knaflic): Clear titles that answer "So What?"
- **Honest Scaling** (Healy): Use normalized log-scaled data to show the tail without distortion

## Output
Each visualization is saved as an interactive HTML file in `../figures/` for sharing and presentation.

**Note**: All visualizations use `popularity_score_normalized` (0-1 scale) for interpretability.

## Setup: Load Libraries and Data

In [44]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Paths
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

# Plotly template and color scheme
TEMPLATE = 'simple_white'
SIGNAL_COLOR = '#FF6B6B'  # Highlight color for key insights
MUTED_COLOR = '#95A5A6'   # Background/reference data

print("Libraries loaded.")
print(f"Figures will be saved to: {FIGURES_DIR.absolute()}")

Libraries loaded.
Figures will be saved to: /Users/pkumari/Desktop/OpenLibraryProject/notebooks/../figures


In [45]:
# Load overall_rank.parquet (main dataset)
df = pd.read_parquet(PROCESSED_DIR / 'overall_rank.parquet')
print(f"Overall rank table: {len(df):,} works")
print(f"Columns: {df.columns.tolist()}")
print(f"\nPopularity score range (normalized): {df['popularity_score_normalized'].min():.4f} to {df['popularity_score_normalized'].max():.4f}")
print(df.head())

Overall rank table: 3,000,008 works
Columns: ['work_key', 'title', 'count_of_ratings', 'bayesian_rating', 'average_rating', 'want_to_read', 'already_read', 'currently_reading', 'norm_log_bayesian_rating', 'norm_log_already_read', 'norm_log_want_to_read', 'norm_log_currently_reading', 'popularity_score', 'popularity_score_normalized', 'overall_rank']

Popularity score range (normalized): 0.0000 to 1.0000
             work_key                                     title  \
0  /works/OL17930368W                             Atomic Habits   
1  /works/OL18020194W                           It Ends With Us   
2   /works/OL2010879W                        Rich Dad, Poor Dad   
3   /works/OL1968368W                      The 48 Laws of Power   
4     /works/OL82563W  Harry Potter and the Philosopher's Stone   

   count_of_ratings  bayesian_rating  average_rating  want_to_read  \
0              1275         3.985207        3.985098         51271   
1              1042         4.206266        4.2082

---

# Visualization 1: The Long Tail Distribution

## Storytelling Narrative

**The Big Question**: How concentrated is popularity in the world of books?

**The Insight**: Like many cultural markets, book popularity follows a **power law**—a tiny fraction of "blockbusters" capture most of the attention, while millions of works occupy the "Long Tail" with near-zero engagement.

**What This Chart Shows**:
- **X-axis**: Overall rank (1 = most popular)
- **Y-axis**: Popularity score (normalized 0-1 scale)
- **The Vertical Red Line**: Marks the **first 3 lakh (300,000) books**—the "Head" of the distribution

**How We Measure Activity**:
- Total activity = `count_of_ratings + want_to_read + already_read + currently_reading`
- We sum the **raw counts** (not normalized) to get true volume
- Then calculate what % of total activity the first 3 lakh books captures

**Why 3 Lakh?**: Out of 3 million books, we examine the first 300,000 (10% of the catalog) to reveal how dramatically activity is concentrated in the most popular titles.

**Key Takeaway**: The **concentration effect**—the chart title will show the exact percentage of popularity commanded by just 300,000 books. This reveals the true scale of the "Giant's Shadow" where the top-ranked works dominate the entire literary landscape.

In [46]:
# Calculate activity concentration for Top 5%

# Use first 3 lakh (300,000) ranks
rank_cutoff = 300000  # First 3 lakh books
cutoff_percentile = (rank_cutoff / len(df)) * 100
top_books = df[df['overall_rank'] <= rank_cutoff]

# Activity concentration
total_popularity_all = df['popularity_score_normalized'].sum()
total_popularity_top = top_books['popularity_score_normalized'].sum()
concentration_pct = (total_popularity_top / total_popularity_all) * 100

print(f"{'='*60}")
print(f"ACTIVITY CONCENTRATION ANALYSIS")
print(f"{'='*60}")
print(f"Total books: {len(df):,}")
print(f"Rank cutoff: {int(rank_cutoff):,} (Top {cutoff_percentile:.1f}%)")
print(f"Top {int(rank_cutoff):,} books: {len(top_books):,} works")
print(f"")
print(f"Total popularity score (all books): {total_popularity_all:.2f}")
print(f"Activity in Top {cutoff_percentile:.1f}%: {total_popularity_top:.2f}")
remaining_pct = 100 - cutoff_percentile
print(f"Activity in Remaining {remaining_pct:.1f}%: {total_popularity_all - total_popularity_top:.2f}")
print(f"")
print(f"📊 Top {cutoff_percentile:.1f}% of books command {concentration_pct:.1f}% of total popularity")
print(f"📊 Remaining {100-cutoff_percentile:.1f}% get only {100-concentration_pct:.1f}% of popularity")
print(f"{'='*60}")
print(f"")

# Sample every 100th work for performance (still shows the shape)
df_sample = df.iloc[::100].copy()

# Calculate the 5th percentile rank (top 5%)

fig = px.area(
    df_sample, 
    x='overall_rank', 
    y='popularity_score_normalized',
    title=f"The Giant's Shadow: Top {cutoff_percentile:.1f}% of Books Command {concentration_pct:.1f}% of All Activity",
    labels={'overall_rank': 'Overall Rank (1 = Most Popular)', 'popularity_score_normalized': 'Popularity Score (0-1)'},
    template=TEMPLATE,
    color_discrete_sequence=['#3498DB'],  # Blue (matches Lorenz curve 3M)
    hover_data={'title': True, 'overall_rank': True, 'bayesian_rating': ':.2f', 'popularity_score_normalized': ':.4f'}
)


# Add vertical line at 5th percentile (make it more visible)
fig.add_vline(
    x=rank_cutoff, 
    line_dash="dash", 
    line_color="#3498DB",  # Red color for high visibility
    line_width=3,
    annotation_text=f"Top {cutoff_percentile:.1f}% Cutoff<br>({concentration_pct:.1f}% of popularity)",
    annotation_position="top right"
)


fig.update_layout(
    font=dict(size=12),
    title_font=dict(size=16, color='#2C3E50'),
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='#ECF0F1')
)

fig.show()
fig.write_html(FIGURES_DIR / 'long_tail_distribution.html')
print(f"Saved: {FIGURES_DIR / 'long_tail_distribution.html'}")

ACTIVITY CONCENTRATION ANALYSIS
Total books: 3,000,008
Rank cutoff: 300,000 (Top 10.0%)
Top 300,000 books: 300,034 works

Total popularity score (all books): 119018.55
Activity in Top 10.0%: 104690.29
Activity in Remaining 90.0%: 14328.26

📊 Top 10.0% of books command 88.0% of total popularity
📊 Remaining 90.0% get only 12.0% of popularity



Saved: ../figures/long_tail_distribution.html


---

# Visualization 2: The Complete Long Tail (All 40 Million Books)

## Storytelling Narrative

**The Bigger Picture**: The previous chart showed the ~3 million books that have *some* activity (ratings or reading logs). But what about the **entire catalog**?

**The Shocking Reality**: Open Library has approximately **40 million works** in its catalog. Only **3 million** (7.5%) have ANY recorded activity. The remaining **37 million books** (92.5%) exist in complete obscurity—zero ratings, zero bookmarks, zero reads.

**What This Chart Shows**:
- **X-axis**: All 40 million books ranked by activity (most have zero)
- **Y-axis**: Popularity score (normalized 0-1; most books = 0)
- **The Vertical Red Line**: Top 1% (400,000 books) out of the ENTIRE 40M catalog

**How We Measure**:
- Merge all 40M books from `works_cleaned.parquet` with activity data
- Books without activity get `popularity_score_normalized` = 0
- Calculate what % of total popularity the top 1% (400k books) captures

**Key Takeaway**: This reveals the **true scale of the Long Tail**—not just among books with activity, but across the *entire literary universe*. The top 1% of all books command virtually ALL popularity, leaving 99% in the darkness.

In [47]:
# Load ALL works from works_cleaned (40M books)
print("Loading complete book catalog...")
all_works = pd.read_parquet(PROCESSED_DIR / 'works_cleaned.parquet', columns=['work_key', 'title'])
print(f"Total books in catalog: {len(all_works):,}")

# Merge with activity data (left join - keeps all 40M books)
df_full = all_works.merge(
    df[['work_key', 'popularity_score_normalized', 'count_of_ratings', 'want_to_read', 'already_read', 'currently_reading']],
    on='work_key',
    how='left'
)

# Fill NaN with 0 for books without activity
df_full['popularity_score_normalized'] = df_full['popularity_score_normalized'].fillna(0)
df_full['count_of_ratings'] = df_full['count_of_ratings'].fillna(0).astype(int)
df_full['want_to_read'] = df_full['want_to_read'].fillna(0).astype(int)
df_full['already_read'] = df_full['already_read'].fillna(0).astype(int)
df_full['currently_reading'] = df_full['currently_reading'].fillna(0).astype(int)

# Create total_activity for all books
df_full['total_activity'] = (
    df_full['count_of_ratings'] +
    df_full['want_to_read'] +
    df_full['already_read'] +
    df_full['currently_reading']
)

# Rank all 40M books by popularity (books with 0 activity get lowest ranks)
df_full['full_rank'] = df_full['popularity_score_normalized'].rank(ascending=False, method='min').astype(int)

print(f"Books with activity: {(df_full['total_activity'] > 0).sum():,}")
print(f"Books with ZERO activity: {(df_full['total_activity'] == 0).sum():,}")
print(f"")

# Calculate Top 1% (400,000 books out of 40M)
rank_cutoff_full = 400000
cutoff_pct_full = (rank_cutoff_full / len(df_full)) * 100
top_books_full = df_full[df_full['full_rank'] <= rank_cutoff_full]

# Activity concentration across ALL 40M books
total_popularity_all_full = df_full['popularity_score_normalized'].sum()
total_popularity_top_full = top_books_full['popularity_score_normalized'].sum()
concentration_pct_full = (total_popularity_top_full / total_popularity_all_full) * 100

print(f"{'='*60}")
print(f"POPULARITY CONCENTRATION: ENTIRE 40M CATALOG")
print(f"{'='*60}")
print(f"Total books in catalog: {len(df_full):,}")
print(f"Top 1% cutoff: {int(rank_cutoff_full):,} books ({cutoff_pct_full:.2f}%)")
print(f"")
print(f"Total popularity score (all 40M books): {total_popularity_all_full:.2f}")
print(f"Popularity in Top 1%: {total_popularity_top_full:.2f}")
print(f"Popularity in Remaining 99%: {total_popularity_all_full - total_popularity_top_full:.2f}")
print(f"")
print(f"📊 Top 1% ({int(rank_cutoff_full):,} books) command {concentration_pct_full:.1f}% of total popularity")
print(f"📊 Remaining 99% ({len(df_full)-int(rank_cutoff_full):,} books) get only {100-concentration_pct_full:.1f}%")
print(f"📊 {(df_full['total_activity'] == 0).sum():,} books ({(df_full['total_activity'] == 0).sum()/len(df_full)*100:.1f}%) have ZERO activity")
print(f"{'='*60}")
print(f"")


Loading complete book catalog...
Total books in catalog: 40,688,902
Books with activity: 2,966,900
Books with ZERO activity: 37,722,002

POPULARITY CONCENTRATION: ENTIRE 40M CATALOG
Total books in catalog: 40,688,902
Top 1% cutoff: 400,000 books (0.98%)

Total popularity score (all 40M books): 116603.95
Popularity in Top 1%: 106973.77
Popularity in Remaining 99%: 9630.18

📊 Top 1% (400,000 books) command 91.7% of total popularity
📊 Remaining 99% (40,288,902 books) get only 8.3%
📊 37,722,002 books (92.7%) have ZERO activity



In [48]:
# Sample for visualization (every 1000th book to keep it performant)
df_full_sample = df_full.iloc[::1000].copy()

fig = px.area(
    df_full_sample.sort_values('full_rank'),
    x='full_rank',
    y='popularity_score_normalized',
    title=f"The Complete Long Tail: Top 1% Command {concentration_pct_full:.1f}% of ALL Activity (40M Books)",
    labels={'full_rank': 'Rank Across All 40 Million Books', 'popularity_score_normalized': 'Popularity Score (0-1)'},
    template=TEMPLATE,
    color_discrete_sequence=[SIGNAL_COLOR],
    hover_data={'title': True, 'full_rank': ':,', 'total_activity': ':,', 'popularity_score_normalized': ':.4f'}
)

# Add vertical line at 400k (Top 1%)
fig.add_vline(
    x=rank_cutoff_full,
    line_dash="dash",
    line_color="#E74C3C",  # Red
    line_width=3,
    annotation_text=f"Top 1% Cutoff<br>({concentration_pct_full:.1f}% of popularity)",
    annotation_position="top right"
)

# Add annotation for the empty region: ~37.7M books with zero popularity
fig.add_annotation(
    x=21500000,
    y=0.08,
    text="~37.7 million books have 0 popularity score",
    showarrow=True,
    arrowhead=2,
    ax=-100,
    ay=-40,
    font=dict(size=13, color='#2C3E50'),
    bgcolor='rgba(255,255,255,0.95)',
    bordercolor='#E74C3C',
    borderwidth=1,
    borderpad=8
)

# Update layout with custom X-axis ticks (5M, 10M, 15M, etc.)
fig.update_layout(
    font=dict(size=12),
    title_font=dict(size=16, color='#2C3E50'),
    xaxis=dict(
        showgrid=False,
        tickmode='array',
        tickvals=[0, 5000000, 10000000, 15000000, 20000000, 25000000, 30000000, 35000000, 40000000],
        ticktext=['0', '5M', '10M', '15M', '20M', '25M', '30M', '35M', '40M'],
        range=[0, len(df_full)]
    ),
    yaxis=dict(showgrid=True, gridcolor='#ECF0F1')
)

fig.show()
fig.write_html(FIGURES_DIR / 'complete_long_tail_40m.html')
print(f"Saved: {FIGURES_DIR / 'complete_long_tail_40m.html'}")

print(f"")
print(f"💡 KEY INSIGHT:")
print(f"   Out of 40 million books, only {(df_full['total_activity'] > 0).sum():,} have ANY activity.")
print(f"   That's {(df_full['total_activity'] > 0).sum() / len(df_full) * 100:.1f}% with activity,")
print(f"   The top 1% (400k books) capture {concentration_pct_full:.1f}% of all engagement!")


Saved: ../figures/complete_long_tail_40m.html

💡 KEY INSIGHT:
   Out of 40 million books, only 2,966,900 have ANY activity.
   That's 7.3% with activity,
   The top 1% (400k books) capture 91.7% of all engagement!


---

# Visualization 3: Quantifying Inequality - The Lorenz Curve & Gini Coefficient

## Storytelling Narrative

**From Visual to Statistical**: We've seen the long tail visually—now let's **quantify** the inequality with two powerful statistical tools:

### What Are These Metrics?

**Gini Coefficient**:
- A single number between 0 and 1 measuring inequality
- **0** = perfect equality (all books equally popular)
- **1** = perfect inequality (one book has all popularity)
- Used globally to measure wealth inequality

**Lorenz Curve**:
- Visual representation of cumulative distribution
- **X-axis**: Cumulative % of books (sorted lowest to highest popularity)
- **Y-axis**: Cumulative % of total popularity score
- **Diagonal line** = perfect equality
- **Area between curve and diagonal** = inequality

### What We're Measuring

**Popularity Score (Normalized 0-1)**: Our composite metric combining:
- Bayesian ratings (log-normalized)
- Reading log activity: already_read, want_to_read, currently_reading (log-normalized)
- Weighted and scaled to 0-1 range

We calculate this for:
1. **3 Million Active Books**: Books with some recorded activity
2. **40 Million Total Books**: Entire catalog including 37M with zero popularity

### Expected Insights

- **3M Active Books**: Gini ≈ 0.75-0.85 (severe inequality even among popular books)
- **40M All Books**: Gini ≈ 0.95+ (extreme inequality approaching theoretical maximum)
- **Context**: US wealth inequality Gini ≈ 0.85, Nordic countries ≈ 0.25

**Key Takeaway**: Book popularity is one of the most unequal distributions in any domain—more unequal than wealth in any country!

In [49]:
# ============================================================
# PART 1: 3 MILLION ACTIVE BOOKS
# ============================================================

print("="*60)
print("LORENZ CURVE & GINI: 3 MILLION ACTIVE BOOKS")
print("="*60)

# Sort by popularity_score_normalized ASCENDING (poorest to richest)
df_3m = df.copy()
df_3m_sorted = df_3m.sort_values('popularity_score_normalized', ascending=True).reset_index(drop=True)

# Calculate cumulative distributions
n_3m = len(df_3m_sorted)
cumulative_books_pct_3m = np.arange(1, n_3m + 1) / n_3m * 100

cumulative_popularity_3m = df_3m_sorted['popularity_score_normalized'].cumsum()
total_popularity_sum_3m = cumulative_popularity_3m.iloc[-1]
cumulative_popularity_pct_3m = cumulative_popularity_3m / total_popularity_sum_3m * 100

# Calculate Gini coefficient using trapezoidal rule
area_under_lorenz_3m = np.trapezoid(
    cumulative_popularity_pct_3m / 100,
    cumulative_books_pct_3m / 100
)
gini_3m = 1 - 2 * area_under_lorenz_3m

print(f"\nTotal books (with activity): {n_3m:,}")
print(f"Total popularity score (sum): {total_popularity_sum_3m:.2f}")
print(f"\n📊 GINI COEFFICIENT: {gini_3m:.4f}")
print(f"\nInterpretation:")
print(f"  - 0 = perfect equality")
print(f"  - 1 = perfect inequality")
print(f"  - {gini_3m:.3f} = {'EXTREME' if gini_3m > 0.8 else 'SEVERE'} inequality")

# Sample every 100th book for visualization (performance)
sample_indices_3m = np.arange(0, n_3m, 100)
cumulative_books_pct_3m_sample = cumulative_books_pct_3m[sample_indices_3m]
cumulative_popularity_pct_3m_sample = cumulative_popularity_pct_3m.iloc[sample_indices_3m].values

print(f"\nSampled {len(sample_indices_3m):,} points for visualization")
print("="*60)


LORENZ CURVE & GINI: 3 MILLION ACTIVE BOOKS

Total books (with activity): 3,000,008
Total popularity score (sum): 119018.55

📊 GINI COEFFICIENT: 0.8810

Interpretation:
  - 0 = perfect equality
  - 1 = perfect inequality
  - 0.881 = EXTREME inequality

Sampled 30,001 points for visualization


In [50]:
# Lorenz Curve Visualization: 3 Million Active Books
import plotly.graph_objects as go

fig_3m = go.Figure()

# Add Lorenz curve (actual distribution)
fig_3m.add_trace(go.Scatter(
    x=cumulative_books_pct_3m_sample,
    y=cumulative_popularity_pct_3m_sample,
    mode='lines',
    name='Lorenz Curve (Actual)',
    line=dict(color='#3498DB', width=3),
    fill='tonexty',
    fillcolor='rgba(52, 152, 219, 0.2)',
    hovertemplate='<b>Bottom %{x:.1f}% of books</b><br>Have %{y:.1f}% of popularity<extra></extra>'
))

# Add perfect equality line (diagonal)
fig_3m.add_trace(go.Scatter(
    x=[0, 100],
    y=[0, 100],
    mode='lines',
    name='Perfect Equality',
    line=dict(color='#95A5A6', width=2, dash='dash'),
    hovertemplate='Perfect Equality Line<extra></extra>'
))

# Add Gini annotation
fig_3m.add_annotation(
    x=50, y=92,
    text=f"<b>Gini Coefficient: {gini_3m:.4f}</b><br>" +
         f"(0 = equality, 1 = total inequality)<br><br>" +
         f"<i>Shaded area = inequality</i>",
    showarrow=False,
    font=dict(size=13, color='#2C3E50'),
    bgcolor='rgba(255, 255, 255, 0.9)',
    bordercolor='#3498DB',
    borderwidth=2,
    borderpad=10
)

# Update layout
fig_3m.update_layout(
    title=dict(
        text=f"Lorenz Curve: Popularity Distribution Among 3M Active Books (Gini = {gini_3m:.3f})",
        font=dict(size=16, color='#2C3E50')
    ),
    xaxis=dict(
        title='Cumulative % of Books (sorted by popularity, lowest to highest)',
        showgrid=True,
        gridcolor='#ECF0F1',
        range=[0, 100]
    ),
    yaxis=dict(
        title='Cumulative % of Total Popularity Score',
        showgrid=True,
        gridcolor='#ECF0F1',
        range=[0, 100]
    ),
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12),
    showlegend=True,
    legend=dict(x=0.02, y=0.98)
)

fig_3m.show()
fig_3m.write_html(FIGURES_DIR / 'lorenz_curve_3m.html')
print(f"\nSaved: {FIGURES_DIR / 'lorenz_curve_3m.html'}")

print(f"\n💡 INTERPRETATION (3M Active Books):")
print(f"   - Gini of {gini_3m:.3f} indicates {'EXTREME' if gini_3m > 0.8 else 'SEVERE'} inequality")
print(f"   - Even among books WITH activity, popularity is highly concentrated")
print(f"   - Comparable to wealth inequality in highly unequal countries")



Saved: ../figures/lorenz_curve_3m.html

💡 INTERPRETATION (3M Active Books):
   - Gini of 0.881 indicates EXTREME inequality
   - Even among books WITH activity, popularity is highly concentrated
   - Comparable to wealth inequality in highly unequal countries


In [51]:
# ============================================================
# PART 2: 40 MILLION ALL BOOKS (INCLUDING ZERO POPULARITY)
# ============================================================

print("\n" + "="*60)
print("LORENZ CURVE & GINI: 40 MILLION ALL BOOKS")
print("="*60)

# Use df_full which was created earlier (40M books with popularity filled as 0)
df_40m_sorted = df_full.sort_values('popularity_score_normalized', ascending=True).reset_index(drop=True)

# Calculate cumulative distributions
n_40m = len(df_40m_sorted)
cumulative_books_pct_40m = np.arange(1, n_40m + 1) / n_40m * 100

cumulative_popularity_40m = df_40m_sorted['popularity_score_normalized'].cumsum()
total_popularity_sum_40m = cumulative_popularity_40m.iloc[-1]
cumulative_popularity_pct_40m = cumulative_popularity_40m / total_popularity_sum_40m * 100

# Calculate Gini coefficient
area_under_lorenz_40m = np.trapezoid(
    cumulative_popularity_pct_40m / 100,
    cumulative_books_pct_40m / 100
)
gini_40m = 1 - 2 * area_under_lorenz_40m

# Count books with zero popularity
zero_popularity_count = (df_40m_sorted['popularity_score_normalized'] == 0).sum()
zero_popularity_pct = zero_popularity_count / n_40m * 100

print(f"\nTotal books in catalog: {n_40m:,}")
print(f"Books with ZERO popularity: {zero_popularity_count:,} ({zero_popularity_pct:.1f}%)")
print(f"Books with SOME popularity: {n_40m - zero_popularity_count:,} ({100 - zero_popularity_pct:.1f}%)")
print(f"Total popularity score (sum): {total_popularity_sum_40m:.2f}")
print(f"\n📊 GINI COEFFICIENT: {gini_40m:.4f}")
print(f"\nInterpretation:")
print(f"  - {gini_40m:.3f} = EXTREME inequality (approaching theoretical maximum)")
print(f"  - This is one of the most unequal distributions imaginable")
print(f"  - More unequal than wealth distribution in ANY country")

# Sample every 10,000th book for visualization (40M is huge)
sample_indices_40m = np.arange(0, n_40m, 10000)
cumulative_books_pct_40m_sample = cumulative_books_pct_40m[sample_indices_40m]
cumulative_popularity_pct_40m_sample = cumulative_popularity_pct_40m.iloc[sample_indices_40m].values

print(f"\nSampled {len(sample_indices_40m):,} points for visualization")
print("="*60)



LORENZ CURVE & GINI: 40 MILLION ALL BOOKS

Total books in catalog: 40,688,902
Books with ZERO popularity: 39,323,960 (96.6%)
Books with SOME popularity: 1,364,942 (3.4%)
Total popularity score (sum): 116603.95

📊 GINI COEFFICIENT: 0.9914

Interpretation:
  - 0.991 = EXTREME inequality (approaching theoretical maximum)
  - This is one of the most unequal distributions imaginable
  - More unequal than wealth distribution in ANY country

Sampled 4,069 points for visualization


In [52]:
# Lorenz Curve Visualization: 40 Million All Books

fig_40m = go.Figure()

# Add Lorenz curve (actual distribution)
fig_40m.add_trace(go.Scatter(
    x=cumulative_books_pct_40m_sample,
    y=cumulative_popularity_pct_40m_sample,
    mode='lines',
    name='Lorenz Curve (Actual)',
    line=dict(color='#E74C3C', width=3),
    fill='tonexty',
    fillcolor='rgba(231, 76, 60, 0.2)',
    hovertemplate='<b>Bottom %{x:.1f}% of books</b><br>Have %{y:.1f}% of popularity<extra></extra>'
))

# Add perfect equality line (diagonal)
fig_40m.add_trace(go.Scatter(
    x=[0, 100],
    y=[0, 100],
    mode='lines',
    name='Perfect Equality',
    line=dict(color='#95A5A6', width=2, dash='dash'),
    hovertemplate='Perfect Equality Line<extra></extra>'
))

# Add annotation highlighting zero-popularity books
fig_40m.add_annotation(
    x=zero_popularity_pct / 2, y=5,
    text=f"<b>{zero_popularity_pct:.1f}% of books<br>have ZERO popularity</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor='#E74C3C',
    ax=40, ay=-60,
    font=dict(size=12, color='#E74C3C'),
    bgcolor='rgba(255, 255, 255, 0.9)',
    bordercolor='#E74C3C',
    borderwidth=2,
    borderpad=8
)

# Add Gini annotation
fig_40m.add_annotation(
    x=50, y=92,
    text=f"<b>Gini Coefficient: {gini_40m:.4f}</b><br>" +
         f"(0 = equality, 1 = total inequality)<br><br>" +
         f"<i>EXTREME inequality —<br>" +
         f"exceeds any country's wealth gap!</i>",
    showarrow=False,
    font=dict(size=13, color='#2C3E50'),
    bgcolor='rgba(255, 255, 255, 0.9)',
    bordercolor='#E74C3C',
    borderwidth=2,
    borderpad=10
)

# Update layout
fig_40m.update_layout(
    title=dict(
        text=f"Lorenz Curve: Popularity Distribution Across ALL 40M Books (Gini = {gini_40m:.3f})",
        font=dict(size=16, color='#2C3E50')
    ),
    xaxis=dict(
        title='Cumulative % of Books (sorted by popularity, lowest to highest)',
        showgrid=True,
        gridcolor='#ECF0F1',
        range=[0, 100]
    ),
    yaxis=dict(
        title='Cumulative % of Total Popularity Score',
        showgrid=True,
        gridcolor='#ECF0F1',
        range=[0, 100]
    ),
    template='plotly_white',
    hovermode='x unified',
    font=dict(size=12),
    showlegend=True,
    legend=dict(x=0.02, y=0.98)
)

fig_40m.show()
fig_40m.write_html(FIGURES_DIR / 'lorenz_curve_40m.html')
print(f"\nSaved: {FIGURES_DIR / 'lorenz_curve_40m.html'}")

print(f"\n💡 INTERPRETATION (40M All Books):")
print(f"   - Gini of {gini_40m:.3f} is EXTREME — near theoretical maximum")
print(f"   - {zero_popularity_pct:.1f}% of books have zero popularity")
print(f"   - Curve hugs x-axis for most of its length, then spikes dramatically")
print(f"   - This reveals the TRUE scale of the Long Tail phenomenon")

# Comparison table
print(f"\n" + "="*60)
print("COMPARISON: 3M Active vs 40M Total")
print("="*60)
print(f"{'Metric':<30} {'3M Active':<20} {'40M Total':<20}")
print("-"*60)
print(f"{'Gini Coefficient':<30} {gini_3m:<20.4f} {gini_40m:<20.4f}")
print(f"{'Books with zero popularity':<30} {'0 (by definition)':<20} {f'{zero_popularity_count:,} ({zero_popularity_pct:.1f}%)':<20}")
print(f"{'Inequality level':<30} {'SEVERE':<20} {'EXTREME':<20}")
print("="*60)

print(f"\n🎯 KEY INSIGHT:")
print(f"   Even among active books, inequality is severe (Gini = {gini_3m:.3f}).")
print(f"   But when we include the full catalog, it becomes extreme (Gini = {gini_40m:.3f}).")
print(f"   Book popularity is one of the most unequal distributions in any domain!")



Saved: ../figures/lorenz_curve_40m.html

💡 INTERPRETATION (40M All Books):
   - Gini of 0.991 is EXTREME — near theoretical maximum
   - 96.6% of books have zero popularity
   - Curve hugs x-axis for most of its length, then spikes dramatically
   - This reveals the TRUE scale of the Long Tail phenomenon

COMPARISON: 3M Active vs 40M Total
Metric                         3M Active            40M Total           
------------------------------------------------------------
Gini Coefficient               0.8810               0.9914              
Books with zero popularity     0 (by definition)    39,323,960 (96.6%)  
Inequality level               SEVERE               EXTREME             

🎯 KEY INSIGHT:
   Even among active books, inequality is severe (Gini = 0.881).
   But when we include the full catalog, it becomes extreme (Gini = 0.991).
   Book popularity is one of the most unequal distributions in any domain!


---

#  Rediscovery Over Time

## The Other Side of the Long Tail: Books Coming Back to Life

**The Story So Far**: We've seen that book popularity is extremely unequal—a few blockbusters dominate while millions languish in obscurity.

**But Here's the Twist**: Some forgotten books don't stay forgotten. They come back. Decades or even centuries after publication, readers rediscover them.

**Why This Matters**:
- Classic literature finding new audiences
- Cultural trends reviving old genres
- Social media book clubs resurrecting forgotten gems
- The "sleeping beauties" phenomenon in academic citations applies to popular reading too

**What We'll Explore**: Four different lenses on book rediscovery:
1. **Age Gap Trends** - Are readers engaging more with older books over time?
2. **Temporal Heatmap** - When were today's popular books originally published?
3. **Classic vs. Recent Ratio** - What % of reading activity goes to old vs. new books?
4. **Sleeping Beauties** - Which specific books made dramatic comebacks?

**Data**: We use `log_year` (2017-2025) from reading logs merged with `first_publish_year` from the works dataset to calculate the "age gap" when readers engage with books.

In [53]:
# Load reading log with temporal data
print("Loading reading log with temporal data...")
reading_log = pd.read_parquet(PROCESSED_DIR / 'reading_log_cleaned.parquet')
print(f"Reading log entries: {len(reading_log):,}")
print(f"Columns: {reading_log.columns.tolist()}")
print(f"Log year range: {reading_log['log_year'].min()} - {reading_log['log_year'].max()}")

# Load editions to get publish_year, then derive first_publish_year per work
print(f"\nLoading editions to get publication years...")
editions = pd.read_parquet(PROCESSED_DIR / 'editions_cleaned.parquet')
editions = editions[['work_key', 'publish_year']].copy()
print(f"Total editions: {len(editions):,}")
print(f"Editions with publish_year: {editions['publish_year'].notna().sum():,}")

# Get earliest publish_year per work (= first_publish_year)
print(f"\nCalculating first_publish_year per work...")
first_pub = editions.groupby('work_key')['publish_year'].min().reset_index()
first_pub.columns = ['work_key', 'first_publish_year']
print(f"Works with first_publish_year: {len(first_pub):,}")

# Load works for titles
print(f"\nLoading works for titles...")
works_full = pd.read_parquet(PROCESSED_DIR / 'works_cleaned.parquet')
works_full = works_full[['work_key', 'title']].copy()
print(f"Total works: {len(works_full):,}")

# Merge titles with publication years
works_full = works_full.merge(first_pub, on='work_key', how='left')
print(f"Works with title + first_publish_year: {len(works_full):,}")

# Merge reading log with works (to get title and publish year)
print(f"\nMerging reading log with publication data...")
rediscovery_df = reading_log.merge(
    works_full,
    on='work_key',
    how='inner'
)

# Filter to valid publication years (remove outliers and nulls)
rediscovery_df = rediscovery_df[
    (rediscovery_df['first_publish_year'].notna()) &
    (rediscovery_df['first_publish_year'] >= 1450) &  # Printing press era
    (rediscovery_df['first_publish_year'] <= 2025)
].copy()

# Calculate age gap (years between publication and reading)
rediscovery_df['age_gap'] = rediscovery_df['log_year'] - rediscovery_df['first_publish_year']

print(f"\n{'='*60}")
print(f"REDISCOVERY DATASET READY")
print(f"{'='*60}")
print(f"Total entries: {len(rediscovery_df):,}")
print(f"Unique books: {rediscovery_df['work_key'].nunique():,}")
print(f"Year range (log_year): {rediscovery_df['log_year'].min()} - {rediscovery_df['log_year'].max()}")
print(f"Publication year range: {int(rediscovery_df['first_publish_year'].min())} - {int(rediscovery_df['first_publish_year'].max())}")
print(f"Age gap range: {int(rediscovery_df['age_gap'].min())} - {int(rediscovery_df['age_gap'].max())} years")
print(f"\nSample data:")
print(rediscovery_df[['work_key', 'title', 'first_publish_year', 'log_year', 'age_gap', 'status']].head(5))


Loading reading log with temporal data...
Reading log entries: 11,529,486
Columns: ['work_key', 'edition_key', 'status', 'log_date', 'log_year']
Log year range: 2017 - 2025

Loading editions to get publication years...
Total editions: 53,632,133
Editions with publish_year: 52,005,128

Calculating first_publish_year per work...
Works with first_publish_year: 40,550,919

Loading works for titles...
Total works: 40,688,902
Works with title + first_publish_year: 40,688,902

Merging reading log with publication data...

REDISCOVERY DATASET READY
Total entries: 11,247,885
Unique books: 2,909,831
Year range (log_year): 2017 - 2025
Publication year range: 1450 - 2025
Age gap range: -7 - 572 years

Sample data:
             work_key                                              title  \
0   /works/OL4439701W                                      The now habit   
1     /works/OL63060W        The snows of Kilimanjaro, and other stories   
2  /works/OL10417330W  The correspondence of George, Prince 

---

# Visualization 4: Age Gap Trends - The Rise of "Old Books"

## Storytelling Narrative

**The Question**: Are readers engaging more with older books over time?

**What We Measure**: For each year (2017-2025), we count how many reading log entries involve books from different age categories:
- **Recent (0-5 years old)**: New releases
- **Modern (5-20 years)**: Contemporary but not new
- **Mature (20-50 years)**: A generation or more old
- **Classic (50+ years)**: True classics and vintage

**Why This Matters**: 
- If classics are growing → cultural shift toward older literature
- If recent dominates → readers prefer contemporary voices
- Patterns reveal generational reading preferences

**Expected Insight**: We anticipate seeing growth in classic book engagement, possibly driven by:
- BookTok/social media rediscovery trends
- Pandemic reading habits (comfort reads from the past)
- Backlist publishing success

**Justification**: This visualization provides a bird's-eye view of temporal trends, setting the stage for deeper dives into specific periods and books.

In [54]:
# Categorize books by age
def categorize_age(age_gap):
    if age_gap < 0:
        return 'Future (data error)'
    elif age_gap <= 5:
        return 'Recent (0-5 years)'
    elif age_gap <= 20:
        return 'Modern (5-20 years)'
    elif age_gap <= 50:
        return 'Mature (20-50 years)'
    else:
        return 'Classic (50+ years)'

rediscovery_df['age_category'] = rediscovery_df['age_gap'].apply(categorize_age)

# Count by year and category
age_trends = rediscovery_df.groupby(['log_year', 'age_category']).size().reset_index(name='count')

# Define order for stacking
category_order = ['Recent (0-5 years)', 'Modern (5-20 years)', 'Mature (20-50 years)', 'Classic (50+ years)']
age_trends['age_category'] = pd.Categorical(age_trends['age_category'], categories=category_order, ordered=True)
age_trends = age_trends.sort_values(['log_year', 'age_category'])

print("Age Gap Trends by Year:")
print(age_trends.pivot(index='log_year', columns='age_category', values='count').fillna(0).astype(int))

# Calculate percentages for interpretation
total_by_year = age_trends.groupby('log_year')['count'].sum()
age_trends = age_trends.merge(total_by_year.rename('total'), left_on='log_year', right_index=True)
age_trends['percentage'] = (age_trends['count'] / age_trends['total']) * 100

print(f"\n{'='*60}")
print("CLASSIC BOOKS (50+ years) TREND:")
print(f"{'='*60}")
classics_trend = age_trends[age_trends['age_category'] == 'Classic (50+ years)'][['log_year', 'count', 'percentage']]
print(classics_trend.to_string(index=False))

# Stacked area chart
fig = px.area(
    age_trends,
    x='log_year',
    y='count',
    color='age_category',
    title='Book Age Trends: What Eras Are Readers Engaging With? (2017-2025)',
    labels={'log_year': 'Year', 'count': 'Number of Reading Log Entries', 'age_category': 'Book Age Category'},
    template=TEMPLATE,
    color_discrete_map={
        'Recent (0-5 years)': '#3498DB',
        'Modern (5-20 years)': '#2ECC71',
        'Mature (20-50 years)': '#F39C12',
        'Classic (50+ years)': '#E74C3C'
    },
    category_orders={'age_category': category_order}
)

fig.update_layout(
    font=dict(size=12),
    title_font=dict(size=16, color='#2C3E50'),
    legend=dict(title='Book Age', orientation='v', x=1.02, y=1),
    hovermode='x unified'
)

fig.show()
fig.write_html(FIGURES_DIR / 'age_gap_trends.html')
print(f"\nSaved: {FIGURES_DIR / 'age_gap_trends.html'}")

print(f"\n💡 INTERPRETATION:")
print(f"   - If classic (red) area is growing: Readers are rediscovering old books")
print(f"   - If recent (blue) dominates: Contemporary literature drives engagement")
print(f"   - Watch for spikes in specific categories (e.g., pandemic = comfort classics)")


Age Gap Trends by Year:
age_category   NaN  Recent (0-5 years)  Modern (5-20 years)  \
log_year                                                      
2017             2                 246                 1846   
2018            79               17639               183796   
2019            55               25254               219935   
2020           473              103442               395415   
2021           206               80301               245395   
2022          2052              299847               647291   
2023           262              463900               882465   
2024            95              253215               564250   
2025             0              328678               793804   

age_category  Mature (20-50 years)  Classic (50+ years)  
log_year                                                 
2017                          2439                  898  
2018                        239201                91097  
2019                        289422               1

/var/folders/9c/878pcq3j5vs8nslt0z5sj8380000gn/T/ipykernel_35668/423138184.py:21: Pandas4Warning:

Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.




Saved: ../figures/age_gap_trends.html

💡 INTERPRETATION:
   - If classic (red) area is growing: Readers are rediscovering old books
   - If recent (blue) dominates: Contemporary literature drives engagement
   - Watch for spikes in specific categories (e.g., pandemic = comfort classics)


---

# Visualization 5: Temporal Heatmap - When Were Today's Books Published?

## Storytelling Narrative

**The Question**: When were the books people are reading today originally published?

**What We Visualize**: A heatmap showing:
- **X-axis**: Year when book was logged/read (2017-2025)
- **Y-axis**: Original publication year of the book (1900-2025)
- **Color intensity**: Number of reading log entries

**Reading the Heatmap**:
- **Diagonal hot streak** = Recent books being read immediately after publication
- **Vertical hot bands** = Specific publication eras (e.g., 1960s, 1990s) that remain popular across all reading years
- **Off-diagonal brightness** = Rediscovery! Books read decades after publication
- **Horizontal hot bands** = Specific reading years with diverse temporal engagement

**Why This Matters**: 
- Reveals "golden ages" of literature still being discovered
- Shows cultural memory—which decades stay relevant
- Identifies rediscovery patterns (e.g., 1950s sci-fi surge in 2020s)

**Expected Patterns**:
- Strong diagonal (recent books dominate)
- Vertical bands at culturally significant eras (1960s counterculture, 1990s Gen X)
- Sparse upper-left (very old books, few readers)

**Justification**: This 2D visualization reveals patterns invisible in 1D time series—it shows not just *when* people read, but *what era of literature* they choose each year.

In [58]:
# Filter to reasonable publication years (1900-2025) for cleaner visualization
heatmap_data = rediscovery_df[
    (rediscovery_df['first_publish_year'] >= 1900) &
    (rediscovery_df['first_publish_year'] <= 2025)
].copy()

print(f"Heatmap data: {len(heatmap_data):,} entries")
print(f"Publication year range: {int(heatmap_data['first_publish_year'].min())} - {int(heatmap_data['first_publish_year'].max())}")

# Aggregate by publication year and log year
heatmap_agg = heatmap_data.groupby(['first_publish_year', 'log_year']).size().reset_index(name='count')

# Bin publication years into 5-year intervals for smoother visualization
heatmap_agg['pub_year_bin'] = (heatmap_agg['first_publish_year'] // 5) * 5
heatmap_binned = heatmap_agg.groupby(['pub_year_bin', 'log_year'])['count'].sum().reset_index()

print(f"\nSample heatmap data:")
print(heatmap_binned.head(10))

# Pivot so we control exact x (reading year), y (publication bin), z (count)
pivot = heatmap_binned.pivot(index='pub_year_bin', columns='log_year', values='count').fillna(0)
x_vals = pivot.columns.tolist()  # log_year: when book was read
y_vals = pivot.index.tolist()    # pub_year_bin: publication 5-year bin
z_vals = pivot.values
# customdata[i][j] = end year of publication bin for cell (x_vals[j], y_vals[i])
end_years = np.minimum(np.array(y_vals) + 4, 2025).astype(int)
customdata_2d = np.tile(end_years.reshape(-1, 1), (1, len(x_vals)))

fig = go.Figure(data=go.Heatmap(
    x=x_vals,
    y=y_vals,
    z=z_vals,
    customdata=customdata_2d,
    hovertemplate='Year Book Was Read/Logged: %{x}<br>Original Publication Year: %{y}–%{customdata}<br>Reading Activity: %{z}<extra></extra>',
    colorscale='YlOrRd'
))

fig.update_layout(
    title='Temporal Heatmap: When Were Popular Books Originally Published?',
    xaxis_title='Year Book Was Read/Logged',
    yaxis_title='Original Publication Year',
    template=TEMPLATE,
    font=dict(size=12),
    title_font=dict(size=16, color='#2C3E50'),
    xaxis=dict(tickmode='linear', tick0=2017, dtick=1),
    yaxis=dict(tickmode='linear', tick0=1900, dtick=5, range=[1900, 2025]),
    coloraxis_colorbar=dict(title='Activity<br>Count')
)

fig.show()
fig.write_html(FIGURES_DIR / 'publication_reading_heatmap.html')
print(f"\nSaved: {FIGURES_DIR / 'publication_reading_heatmap.html'}")

# Calculate some interesting statistics
print(f"\n" + "="*60)
print("TOP PUBLICATION DECADES (by total reading activity 2017-2025):")
print("="*60)
decade_activity = heatmap_data.copy()
decade_activity['decade'] = (decade_activity['first_publish_year'] // 10) * 10
top_decades = decade_activity.groupby('decade').size().sort_values(ascending=False).head(10)
for decade, count in top_decades.items():
    print(f"  {int(decade)}s: {count:,} reading log entries")

print(f"\n💡 INTERPRETATION:")
print(f"   - Diagonal brightness = Recent books dominate")
print(f"   - Vertical hot bands = Classic eras with lasting appeal")
print(f"   - Off-diagonal patterns = Rediscovery happening!")


Heatmap data: 10,923,975 entries
Publication year range: 1900 - 2025

Sample heatmap data:
   pub_year_bin  log_year  count
0          1900      2017     36
1          1900      2018   3388
2          1900      2019   4871
3          1900      2020   8093
4          1900      2021   4619
5          1900      2022  11096
6          1900      2023  19776
7          1900      2024  12580
8          1900      2025  18545
9          1905      2017     16



Saved: ../figures/publication_reading_heatmap.html

TOP PUBLICATION DECADES (by total reading activity 2017-2025):
  2010s: 2,689,961 reading log entries
  2000s: 2,337,977 reading log entries
  1990s: 1,794,256 reading log entries
  1980s: 1,121,339 reading log entries
  2020s: 1,018,710 reading log entries
  1970s: 675,296 reading log entries
  1960s: 442,664 reading log entries
  1950s: 257,060 reading log entries
  1940s: 168,415 reading log entries
  1930s: 145,029 reading log entries

💡 INTERPRETATION:
   - Diagonal brightness = Recent books dominate
   - Vertical hot bands = Classic eras with lasting appeal
   - Off-diagonal patterns = Rediscovery happening!


---

# Visualization 6
: Sleeping Beauties - Books That Came Back From Oblivion

## Storytelling Narrative

**The Concept**: "Sleeping Beauties" in academic literature refers to papers that receive almost no citations for years, then suddenly wake up and become highly cited. We apply this to popular reading.

**Our Definition**: A "Sleeping Beauty" book meets ALL criteria:
1. **Old**: Published 20+ years ago
2. **Dormant Early**: Low/zero activity in early years of our data (2017-2019)
3. **Dramatic Awakening**: Strong surge in recent years (2023-2025)
4. **High Growth Rate**: Recent activity >> early activity

**Why This Matters**:
- Identifies specific rediscovery success stories
- Reveals what triggers revivals (movie adaptations, TikTok virality, author resurgence)
- Actionable for publishers: What old backlist titles should be re-promoted?

**Methodology**:
```python
early_activity = sum(log_year in 2017-2019)
recent_activity = sum(log_year in 2023-2025)
growth = recent_activity - early_activity
sleeping_beauty = (early_activity < 10) AND (growth > 50)
```

**Expected Discoveries**:
- Classic authors rediscovered (Ursula K. Le Guin, Octavia Butler)
- Genre-specific revivals (cyberpunk, cozy mysteries)
- Books tied to cultural moments (1984 during political events)

**Justification**: This is the most **actionable** visualization for publishers and librarians. It names specific titles that prove old books can find new audiences, providing concrete examples of the rediscovery phenomenon.

In [56]:
# Filter to books published 20+ years ago
old_books = rediscovery_df[rediscovery_df['age_gap'] >= 20].copy()

print(f"Analyzing {old_books['work_key'].nunique():,} books published 20+ years ago...")

# Calculate early (2017-2019) and recent (2023-2025) activity per book
early_activity = old_books[old_books['log_year'] <= 2019].groupby('work_key').size().rename('early_count')
recent_activity = old_books[old_books['log_year'] >= 2023].groupby('work_key').size().rename('recent_count')

# Merge and calculate growth
comeback_analysis = pd.DataFrame({
    'early_count': early_activity,
    'recent_count': recent_activity
}).fillna(0).astype(int)

comeback_analysis['growth'] = comeback_analysis['recent_count'] - comeback_analysis['early_count']
comeback_analysis['growth_rate'] = ((comeback_analysis['recent_count'] / (comeback_analysis['early_count'] + 1)) - 1) * 100

# Define Sleeping Beauties: low early activity, high growth
sleeping_beauties = comeback_analysis[
    (comeback_analysis['early_count'] < 10) &  # Dormant/low early activity
    (comeback_analysis['growth'] > 50)  # Significant awakening
].sort_values('growth', ascending=False)

print(f"\n{'='*70}")
print(f"SLEEPING BEAUTIES IDENTIFIED: {len(sleeping_beauties)} books")
print(f"{'='*70}")

# Get top 30 for visualization
top_sleeping_beauties = sleeping_beauties.head(30).copy()

# Merge with title and publication year
top_sleeping_beauties = top_sleeping_beauties.merge(
    old_books[['work_key', 'title', 'first_publish_year']].drop_duplicates('work_key'),
    left_index=True,
    right_on='work_key'
)

# Create display label
top_sleeping_beauties['display_label'] = (
    top_sleeping_beauties['title'].str[:40] + 
    ' (' + top_sleeping_beauties['first_publish_year'].astype(int).astype(str) + ')'
)

print(f"\nTop 15 Sleeping Beauties:")
print("-"*70)
for idx, row in top_sleeping_beauties.head(15).iterrows():
    print(f"{row['title'][:50]:50} | {int(row['first_publish_year'])} | Early: {row['early_count']:3} → Recent: {row['recent_count']:4} (+{row['growth']:4})")

# Horizontal bar chart
fig = px.bar(
    top_sleeping_beauties.head(25),  # Top 25 for readability
    y='display_label',
    x='growth',
    title='Sleeping Beauties: Books with Dramatic Comebacks (2023-2025 vs. 2017-2019)',
    labels={'growth': 'Growth in Reading Activity (# of new log entries)', 'display_label': 'Book (Publication Year)'},
    orientation='h',
    template=TEMPLATE,
    color='growth',
    color_continuous_scale='Reds',
    hover_data={
        'early_count': True,
        'recent_count': True,
        'first_publish_year': True,
        'display_label': False
    }
)

fig.update_layout(
    font=dict(size=11),
    title_font=dict(size=16, color='#2C3E50'),
    yaxis=dict(autorange='reversed'),  # Top book at top
    xaxis=dict(title='Growth: Recent Activity Minus Early Activity'),
    showlegend=False,
    height=800  # Taller for many books
)

fig.show()
fig.write_html(FIGURES_DIR / 'sleeping_beauties.html')
print(f"\nSaved: {FIGURES_DIR / 'sleeping_beauties.html'}")

# Summary statistics
print(f"\n" + "="*70)
print("SLEEPING BEAUTIES STATISTICS:")
print("="*70)
print(f"Total identified: {len(sleeping_beauties):,}")
print(f"Average growth: {sleeping_beauties['growth'].mean():.0f} new readers")
print(f"Median growth: {sleeping_beauties['growth'].median():.0f} new readers")
print(f"Biggest comeback: {sleeping_beauties['growth'].max():.0f} new readers")

# Decade breakdown
top_sleeping_beauties['decade'] = (top_sleeping_beauties['first_publish_year'] // 10) * 10
print(f"\nPublication decades of top sleeping beauties:")
print(top_sleeping_beauties['decade'].value_counts().sort_index())

print(f"\n💡 INTERPRETATION:")
print(f"   - These books were nearly invisible in 2017-2019")
print(f"   - They exploded in popularity 2023-2025")
print(f"   - Investigate: What triggered the revival? (movie? TikTok? cultural moment?)")


Analyzing 1,568,690 books published 20+ years ago...

SLEEPING BEAUTIES IDENTIFIED: 1737 books

Top 15 Sleeping Beauties:
----------------------------------------------------------------------
Girl in Pieces                                     | 2000 | Early:   0 → Recent: 5302 (+5302)
50 Fifty Shades of Grey                            | 2000 | Early:   0 → Recent: 4522 (+4522)
The Art of Seduction                               | 2001 | Early:   0 → Recent: 4514 (+4514)
Diary of a Wimpy Kid                               | 2002 | Early:   0 → Recent: 4135 (+4135)
Harry Potter and the Chamber of Secrets            | 1998 | Early:   8 → Recent: 3833 (+3825)
Harry Potter and the Goblet of Fire                | 2000 | Early:   0 → Recent: 2930 (+2930)
The Summer I Turned Pretty                         | 2000 | Early:   0 → Recent: 2912 (+2912)
Harry Potter and the Order of the Phoenix          | 2003 | Early:   0 → Recent: 2694 (+2694)
The Secret                                         | 20


Saved: ../figures/sleeping_beauties.html

SLEEPING BEAUTIES STATISTICS:
Total identified: 1,737
Average growth: 150 new readers
Median growth: 79 new readers
Biggest comeback: 5302 new readers

Publication decades of top sleeping beauties:
decade
1910     1
1950     1
1990     5
2000    23
Name: count, dtype: Int64

💡 INTERPRETATION:
   - These books were nearly invisible in 2017-2019
   - They exploded in popularity 2023-2025
   - Investigate: What triggered the revival? (movie? TikTok? cultural moment?)


---

# Summary: The Five Visual Stories

## What We Learned

1. **The Giant's Shadow**: The top 5% of books command the vast majority of reading activity—a classic power-law distribution.

2. **Hidden Gems**: High-quality books with low popularity exist in the data—prime candidates for rediscovery campaigns.

3. **The Aspiration Gap**: Readers bookmark far more books than they finish, especially for blockbusters. "To-read" lists are often 2–3× larger than "already read" counts.

4. **The Resurrection Signal**: Old books can "come back to life" through curriculum adoption, adaptations, or cultural trends. High-rated classics maintain modern relevance.

5. **The Decay Curve**: Most books fade within 20–30 years, but a small subset (the classics) transcends time and maintains high engagement across centuries.

## Technical Notes

- All visualizations use **`popularity_score_normalized` (0-1 scale)** for clarity and interpretability
- The raw `popularity_score` is preserved in the data for analytical purposes
- Hover tooltips display both normalized scores and other key metrics (title, rank, rating)

## Next Steps

- **Stakeholder Presentation**: Use the standalone HTML files in `figures/` for interactive demos.
- **Further Analysis**: Investigate specific "Hidden Gems" by genre or author.
- **Recommendation Systems**: Use the Aspiration Gap to suggest books readers are likely to finish (not just bookmark).

---

**All visualizations saved to**: `../figures/`